In [1]:
import pandas as pd
import numpy as np
import os
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from scipy.signal import savgol_filter

In [2]:
# ==========================================
# 1. METRICS CALCULATION FUNCTIONS
# ==========================================
def calculate_metrics(original, denoised):
    """
    Calculates standard signal quality metrics.
    """
    mse = np.mean((original - denoised) ** 2)
    # Signal-to-Noise Ratio (SNR) in dB
    snr = 10 * np.log10(np.sum(original**2) / np.sum((original - denoised)**2))
    # Percentage Root-Mean-Square Difference (PRD)
    prd = np.sqrt(np.sum((original - denoised)**2) / np.sum(original**2)) * 100
    return {"MSE": mse, "SNR_dB": snr, "PRD": prd}

In [ ]:
def calculate_metrics1(original, denoised):
    """
    Calculates standard signal quality metrics with DC-offset correction.
    """
    # 1. Remove Mean for Metric Calculation (AC-coupled SNR)
    # This prevents a massive SNR value just because the baseline is -5.8
    orig_ac = original - np.mean(original)
    denoised_ac = denoised - np.mean(denoised)
    
    mse = np.mean((original - denoised) ** 2)
    
    # Signal-to-Noise Ratio (SNR) in dB using AC components
    noise_power = np.sum((orig_ac - denoised_ac)**2)
    signal_power = np.sum(orig_ac**2)
    snr_db = 10 * np.log10(signal_power / (noise_power + 1e-12))
    
    # 2. PRD (Percentage Root-Mean-Square Difference)
    # Lower is better; measures distortion relative to the original signal
    prd = np.sqrt(np.sum((original - denoised)**2) / np.sum(original**2)) * 100
    
    return {"MSE": mse, "SNR_dB": snr_db, "PRD": prd}

In [3]:
# ==========================================
# 2. NOVEL STRD PIPELINE
# ==========================================
def strd_pipeline(df, pca_components=2, window_size=51):
    data = df.values
    df_temp = pd.DataFrame(data)
    
    # Stage 1: Adaptive Despiking (Temporal)
    rolling_mean = df_temp.rolling(window=11, center=True).mean()
    rolling_std = df_temp.rolling(window=11, center=True).std()
    z_scores = np.abs((df_temp - rolling_mean) / (rolling_std + 1e-6))
    data_despiked = np.where(z_scores > 3, rolling_mean, df_temp)
    data_despiked = pd.DataFrame(data_despiked).fillna(method='bfill').fillna(method='ffill').values

    # Stage 2: Differential Mode PCA (Spatial)
    pca = PCA(n_components=pca_components)
    scores = pca.fit_transform(data_despiked)
    
    # Smooth dominant modes
    scores_filtered = np.zeros_like(scores)
    for i in range(pca_components):
        # Using Savitzky-Golay to preserve peak information
        scores_filtered[:, i] = savgol_filter(scores[:, i], window_length=window_size, polyorder=3)
        
    signal_core = pca.inverse_transform(scores_filtered)
    
    # Stage 3: Information-Weighted Recombination (Adaptive)
    activity = df_temp.rolling(window=21, center=True).var().mean(axis=1).values
    activity = np.nan_to_num(activity)
    # Normalize weight to adjust cleaning intensity
    weight = (activity - np.min(activity)) / (np.max(activity) - np.min(activity) + 1e-6)
    weight = np.clip(weight * 0.5, 0, 0.5).reshape(-1, 1)
    
    final_output = (1 - weight) * signal_core + weight * data_despiked
    return final_output

In [4]:
# Plot the results
def plot_initial_vs_final(df_raw, final, i, output_root):
    """
    Plots and saves the initial (raw) data against the final output data.
    """
    plt.figure(figsize=(12, 6))
    
    # Plot first 1000 samples
    plt.plot(df_raw.iloc[:1000, 0], label='Initial (Raw)', color='red', alpha=0.7)
    plt.plot(final[:1000, 0], label='Final Output', color='blue', linewidth=1.5)
    
    plt.title(f"Initial vs. Final Comparison - File {i}")
    plt.xlabel("Sample Index")
    plt.ylabel("Amplitude")
    plt.legend(loc='upper right')
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    
    # Save the plot
    plt.savefig(f"{output_root}comparison_plots/initial_vs_final_{i}.png")
    plt.close()


In [5]:
from scipy.signal import butter, filtfilt

def zero_phase_butterworth(data, cutoff, fs, order=5):
    b, a = butter(order, cutoff / (0.5 * fs), btype='low')
    # filtfilt performs the forward-backward pass to eliminate phase shift
    return filtfilt(b, a, data)

In [9]:
def strd_pipeline1(df, pca_components=2, window_size=15): # Reduced window size
    data = df.values
    # Save the mean to prevent amplitude shifting
    column_means = np.mean(data, axis=0)
    data_centered = data - column_means
    
    df_temp = pd.DataFrame(data_centered)
    
    # --- Stage 1: Robust Despiking (Non-destructive) ---
    # Use Median instead of Mean to preserve sharp transitions
    rolling_median = df_temp.rolling(window=7, center=True).median()
    rolling_mad = (df_temp - rolling_median).abs().rolling(window=7, center=True).median()
    # Hampel Filter logic: replace only outliers (>3 * 1.4826 * MAD)
    threshold = 3 * 1.4826 * rolling_mad
    outliers = (df_temp - rolling_median).abs() > threshold
    
    data_despiked = np.where(outliers, rolling_median, df_temp)
    data_despiked = pd.DataFrame(data_despiked).fillna(method='bfill').fillna(method='ffill').values

    # --- Stage 2: PCA with Order Preservation ---
    pca = PCA(n_components=pca_components)
    scores = pca.fit_transform(data_despiked)
    
    scores_filtered = np.zeros_like(scores)
    for i in range(pca_components):
        # Reduced window_length and increased polyorder to 3-5 to keep peaks sharp
        scores_filtered[:, i] = savgol_filter(scores[:, i], window_length=window_size, polyorder=3)
        
    signal_core = pca.inverse_transform(scores_filtered)
    
    # --- Stage 3: Adaptive Recombination ---
    # We use the despiked data as the "truth" for peaks
    final_output = signal_core + column_means # Add mean back here
    
    return final_output

In [10]:
# ==========================================
# 3. BATCH PROCESSING & PLOTTING
# ==========================================
input_folder = 'csv/'  # Your folder with 40 files
output_folder = 'results/'
os.makedirs(output_folder, exist_ok=True)

all_stats = []

# Loop through files 1 to 40
for i in range(1, 41):
    file_path = f"{input_folder}{i}_raw.csv"
    
    if os.path.exists(file_path):
        print(f"Processing: {file_path}")
        
        # A. LOAD
        df_raw = pd.read_csv(file_path, header=None)
        raw_mean = np.mean(df_raw, axis=0)
        data_centered = df_raw - raw_mean
        
        # B. DENOISE (using STRD)
        denoised_data = strd_pipeline1(data_centered)
        df_denoised = pd.DataFrame(denoised_data, columns=df_raw.columns)
        final_output = df_denoised + raw_mean
        
        # C. METRICS (Calculate for the first column as reference)
        stats = calculate_metrics(df_raw[0].values, final_output[0].values)
        stats['File'] = f"{i}_raw.csv"
        all_stats.append(stats)
        
        # D. PLOT (Save visual comparison for first 1000 samples)
        # plot_initial_vs_final(df_raw, df_denoised.iloc[:, 0].values, i, output_folder)
        plt.figure(figsize=(12, 6))
        plt.plot(df_raw.iloc[:1000, 0], label='Original (Raw)', alpha=0.4, color='gray')
        plt.plot(final_output.iloc[:1000, 0], label='STRD Denoised', color='blue', linewidth=1.5)
        plt.title(f"Denoising Result - File {i} (Col 0)")
        plt.legend()
        plt.grid(True, linestyle='--', alpha=0.7)
        plt.savefig(f"{output_folder}plot_file_{i}.png")
        plt.close()
        
        # E. SAVE CLEANED DATA
        df_denoised.to_csv(f"{output_folder}{i}_cleaned.csv", index=False)

Processing: csv/1_raw.csv
Processing: csv/2_raw.csv
Processing: csv/3_raw.csv
Processing: csv/4_raw.csv
Processing: csv/5_raw.csv
Processing: csv/6_raw.csv
Processing: csv/7_raw.csv
Processing: csv/8_raw.csv
Processing: csv/9_raw.csv
Processing: csv/10_raw.csv
Processing: csv/11_raw.csv
Processing: csv/12_raw.csv
Processing: csv/13_raw.csv
Processing: csv/14_raw.csv
Processing: csv/15_raw.csv
Processing: csv/16_raw.csv
Processing: csv/17_raw.csv
Processing: csv/18_raw.csv
Processing: csv/19_raw.csv
Processing: csv/20_raw.csv
Processing: csv/21_raw.csv
Processing: csv/22_raw.csv
Processing: csv/23_raw.csv
Processing: csv/24_raw.csv
Processing: csv/25_raw.csv
Processing: csv/26_raw.csv
Processing: csv/27_raw.csv
Processing: csv/28_raw.csv
Processing: csv/29_raw.csv
Processing: csv/30_raw.csv
Processing: csv/31_raw.csv
Processing: csv/32_raw.csv
Processing: csv/33_raw.csv
Processing: csv/34_raw.csv
Processing: csv/35_raw.csv
Processing: csv/36_raw.csv
Processing: csv/37_raw.csv
Processing

In [ ]:
# ==========================================
# 4. FINAL SUMMARY
# ==========================================
summary_df = pd.DataFrame(all_stats)
summary_df.to_csv(f"{output_root}performance_metrics.csv", index=False)
print("\nProcessing Complete.")
print(summary_df.head())


Processing Complete.
        MSE     SNR_dB        PRD       File
0  0.003495  42.480812   0.751553  1_raw.csv
1  0.003924  38.103058   1.244077  2_raw.csv
2  0.097706   5.972957  50.275010  3_raw.csv
3  0.005423  30.955122   2.832983  4_raw.csv
4  0.004488  24.842807   5.726110  5_raw.csv


: 